<a href="https://colab.research.google.com/github/benjibrcz/deep-ltl-interp/blob/main/Timaeus_2026_Research_Scientist_Work_Test_(In_context_learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Timaeus \- Research Scientist \- Work Test 2026**

### **Background**

Large language models exhibit **in-context learning (ICL)**: they improve at prediction as they see more tokens within a single context, *without any weight updates*. There are several theoretical perspectives:

1. **ICL as in-context supervised learning**: Given question-answer pairs in the prompt, the model becomes more accurate at answering later questions.  
2. **ICL as in-context empirical risk minimization**: ICL occurs if the per-token loss decreases with context length.  
3. **ICL as in-context Bayesian inference**: ICL can be understood as approximate Bayesian inference over latent concepts or tasks.

You are **not expected** to be familiar with this literature.

---

### **Your Task**

Design and implement a prototype evaluation method to assess the in-context learning capabilities of transformers.

**Guiding questions:**

* What does it mean for a model to "learn" in-context? What does it mean for a model to *not* use in-context learning? What phenomena is ICL distinct from?   
* What tasks could reveal ICL? What properties should they have?  
* How do you measure ICL performance?  
* How do you control for confounds?

This is deliberately open-ended: you can focus on evaluating pretrained language models or on evaluating small transformers that you personally train from scratch on synthetic tasks; you can focus on evaluating a single model in detail or doing a broad comparative analysis; you can assume one of the three theoretical perspectives above and continue from there, or you can focus on comparing the three perspectives; etc. There is no "correct" way to approach this problem, and we recommend you spend some time thinking through options yourself. When you find yourself making choices about direction, please explain why you chose to focus on X over Y.

If you find yourself stuck, the references at the end of this document point to related work that may spark ideas. Please do not spend your entire work test reading papers.

---

### **What we’re looking for:**

* **Research thinking**: Clear problem formulation, awareness of confounds, thoughtful design  
* **Technical execution**: Working code, appropriate methods, correct implementation  
* **Communication**: Clear documentation, well-organized notebooks, interpretable results  
* **Depth vs. breadth**: Good decisions about what to pursue given time constraints

Feel free to decide to go deeper in some areas at the expense of others.


# 1. Design

## Overview

We evaluate ICL by presenting models with **k demonstration pairs** `(x_i, f(x_i))` followed by a
**test query** `x_test`, and measuring how well the model predicts `f(x_test)` as k increases.

## Model: Pythia family (EleutherAI)
- Available in sizes 70M → 6.9B (scaling analysis)
- Freely available, Colab-friendly
- Well-studied in ICL literature

## Tasks (synthetic, novel, ground-truth available)
1. **Arbitrary symbol mapping**: word → random integer
2. **Linear functions**: y = ax + b
3. **Modular arithmetic**: (a+b) mod p

## Confound controls (5 conditions per task)
| Condition | Description | Controls for |
|-----------|-------------|-------------|
| Standard | k correct demos | Baseline ICL |
| Irrelevant demos | k demos from different rule | "More context = better" |
| Shuffled labels | Same inputs, random outputs | Format imitation |
| Reversed order | Correct demos, reversed | Order sensitivity |
| Recency conflict | First half correct, second half wrong rule | Recency bias |

## Metrics (3 theoretical perspectives)
1. **Supervised learning**: Accuracy vs k
2. **ERM**: Per-token loss at answer position vs context length
3. **Bayesian**: Learning curve shape, permutation sensitivity

In [ ]:
# Setup and imports
!pip install -q transformers accelerate

import torch
import numpy as np
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Load model and tokenizer
# Start with Pythia-410M as a reasonable middle ground for testing.
# Can scale up/down later for comparative analysis.

MODEL_NAME = "EleutherAI/pythia-410m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)
model.eval()

print(f"Loaded {MODEL_NAME} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params)")

# 2. Metrics

Three metrics corresponding to the three theoretical perspectives on ICL:

1. **Accuracy vs k** (supervised learning view): Does the model get the right answer more often with more demos?
2. **Answer loss vs k** (ERM view): Does the cross-entropy loss on the answer tokens decrease with more demos?
3. **Bayesian diagnostics**: Is the learning curve log-linear? Is performance invariant to demo order?

In [ ]:
@torch.no_grad()
def evaluate_prompt(model, tokenizer, prompt: str, target: str, device) -> dict:
    """Evaluate a single prompt+target pair.

    Returns:
        dict with:
            - 'accuracy': 1.0 if greedy-decoded answer matches target, else 0.0
            - 'loss': cross-entropy loss on the target tokens
            - 'generated': the model's greedy-decoded answer (for debugging)
    """
    # Tokenize prompt and target separately so we know where the answer starts
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    target_ids = tokenizer.encode(target, add_special_tokens=False, return_tensors="pt").to(device)
    full_ids = torch.cat([prompt_ids, target_ids], dim=1)

    # Forward pass on full sequence
    outputs = model(full_ids)
    logits = outputs.logits  # (1, seq_len, vocab)

    # Loss on target tokens only:
    # logits at position [prompt_len-1 .. prompt_len+target_len-2] predict
    # tokens at position [prompt_len .. prompt_len+target_len-1]
    prompt_len = prompt_ids.shape[1]
    target_len = target_ids.shape[1]

    target_logits = logits[0, prompt_len - 1 : prompt_len + target_len - 1, :]  # (target_len, vocab)
    target_tokens = target_ids[0]  # (target_len,)

    loss = torch.nn.functional.cross_entropy(target_logits.float(), target_tokens).item()

    # Greedy decode: generate target_len tokens from prompt
    generated_ids = model.generate(
        prompt_ids,
        max_new_tokens=target_len,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(generated_ids[0, prompt_len:], skip_special_tokens=True).strip()
    accuracy = 1.0 if generated_text == target.strip() else 0.0

    return {
        "accuracy": accuracy,
        "loss": loss,
        "generated": generated_text,
    }


# Quick sanity check
result = evaluate_prompt(model, tokenizer, "The capital of France is", " Paris", device)
print(f"Sanity check - generated: '{result['generated']}', loss: {result['loss']:.3f}, acc: {result['accuracy']}")

In [ ]:
# 3. Datasets

## Task 1: Arbitrary Symbol Mapping

**Setup**: Map common English words to random single-digit integers (0-9).
Each "task instance" is a fresh random mapping. The model sees k demo pairs
like `"apple -> 7\nbanana -> 3\n"` and must predict the output for a held-out word.

**Why single digits?** Keeps the target to a single token, making accuracy/loss
clean to measure. The mapping is completely arbitrary — no semantic relationship
between words and numbers — so the model can't use pretraining knowledge.

**Word pool**: We use 50 common, unambiguous English nouns. For each task instance,
we sample a subset and assign random labels.

In [ ]:
# Word pool: common, concrete, unambiguous English nouns
WORD_POOL = [
    "apple", "tiger", "river", "piano", "cloud", "bread", "chair", "flame",
    "grape", "horse", "knife", "lemon", "mouse", "ocean", "pearl", "queen",
    "robot", "snake", "torch", "whale", "arrow", "badge", "candy", "drum",
    "eagle", "fence", "globe", "heart", "ivory", "jewel", "koala", "lunar",
    "maple", "nerve", "olive", "pilot", "quilt", "reign", "storm", "tower",
    "union", "vapor", "wrist", "yacht", "zebra", "brick", "crane", "delta",
    "frost", "grain",
]

# K values to test (extended range)
K_VALUES = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

# Labels for multi-class tasks (single tokens, alphabetical)
CLASS_LABELS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")


class SymbolMappingTask:
    """Generates task instances for arbitrary word -> digit mapping."""

    def __init__(self, word_pool=WORD_POOL, n_labels=10, seed=None):
        self.word_pool = word_pool
        self.n_labels = n_labels
        self.rng = np.random.RandomState(seed)

    def sample_task(self, n_demos: int, n_test: int = 1):
        n_total = n_demos + n_test
        words = self.rng.choice(self.word_pool, size=n_total, replace=False).tolist()
        labels = self.rng.randint(0, self.n_labels, size=n_total).tolist()
        pairs = [(w, str(l)) for w, l in zip(words, labels)]
        return pairs[:n_demos], pairs[n_demos:]

    def sample_distractor_task(self, n_demos: int):
        words = self.rng.choice(self.word_pool, size=n_demos, replace=False).tolist()
        labels = self.rng.randint(0, self.n_labels, size=n_demos).tolist()
        return [(w, str(l)) for w, l in zip(words, labels)]


class MultiThresholdTask:
    """N-class threshold classification with controllable difficulty.

    Partitions numbers 0..(num_range-1) into n_classes intervals using
    (n_classes - 1) random thresholds. Each interval maps to a different
    class label (A, B, C, ...).

    Difficulty is controlled by n_classes:
      - n_classes=2: one threshold, binary (easy)
      - n_classes=4: three thresholds (medium)
      - n_classes=8: seven thresholds (hard)
      - n_classes=16: fifteen thresholds (very hard)

    Thresholds are constrained so each class covers at least 5% of the range,
    ensuring every class is represented in the demos.
    """

    def __init__(self, n_classes=2, num_range=1000, seed=None):
        self.n_classes = n_classes
        self.num_range = num_range
        self.labels = CLASS_LABELS[:n_classes]
        self.rng = np.random.RandomState(seed)

    def _sample_thresholds(self):
        """Sample n_classes-1 sorted thresholds with minimum spacing."""
        min_gap = max(self.num_range // (self.n_classes * 4), 2)  # ~5% of range per class minimum
        # Sample thresholds with rejection to ensure spacing
        for _ in range(1000):
            raw = sorted(self.rng.randint(min_gap, self.num_range - min_gap,
                                          size=self.n_classes - 1).tolist())
            # Check minimum gaps
            points = [0] + raw + [self.num_range]
            gaps = [points[i+1] - points[i] for i in range(len(points)-1)]
            if all(g >= min_gap for g in gaps):
                return raw
        # Fallback: evenly spaced
        step = self.num_range // self.n_classes
        return [step * (i + 1) for i in range(self.n_classes - 1)]

    def _classify(self, x, thresholds):
        """Classify a number given sorted thresholds."""
        for i, t in enumerate(thresholds):
            if x < t:
                return self.labels[i]
        return self.labels[-1]

    def sample_task(self, n_demos: int, n_test: int = 1):
        thresholds = self._sample_thresholds()
        n_total = n_demos + n_test
        xs = self.rng.choice(self.num_range, size=n_total, replace=False).tolist()
        pairs = [(str(x), self._classify(x, thresholds)) for x in xs]
        return pairs[:n_demos], pairs[n_demos:]

    def sample_distractor_task(self, n_demos: int):
        thresholds = self._sample_thresholds()  # different thresholds
        xs = self.rng.choice(self.num_range, size=n_demos, replace=False).tolist()
        return [(str(x), self._classify(x, thresholds)) for x in xs]


# Backwards compatibility: BinaryThresholdTask is just MultiThresholdTask with n_classes=2
def BinaryThresholdTask(num_range=1000, seed=None):
    return MultiThresholdTask(n_classes=2, num_range=num_range, seed=seed)


def format_demos(demos: list, test_word: str) -> str:
    """Format demo pairs + test query into a prompt string."""
    lines = [f"{word} -> {label}" for word, label in demos]
    lines.append(f"{test_word} ->")
    return "\n".join(lines)


# Quick tests
print("=== Symbol Mapping Task ===")
task_gen = SymbolMappingTask(seed=0)
demos, tests = task_gen.sample_task(n_demos=4, n_test=1)
print(format_demos(demos, tests[0][0]))
print(f"Expected: {tests[0][1]}\n")

print("=== Binary (2-class) ===")
task2 = MultiThresholdTask(n_classes=2, seed=0)
demos2, tests2 = task2.sample_task(n_demos=6, n_test=1)
print(format_demos(demos2, tests2[0][0]))
print(f"Expected: {tests2[0][1]}\n")

print("=== 4-class ===")
task4 = MultiThresholdTask(n_classes=4, seed=0)
demos4, tests4 = task4.sample_task(n_demos=8, n_test=1)
print(format_demos(demos4, tests4[0][0]))
print(f"Expected: {tests4[0][1]}")

In [ ]:
# 3.5 Confound Capability Benchmarks

Before testing for ICL, we verify the model **can already do** the simpler behaviors
that might explain apparent learning. If these are near-ceiling at low k, then any
ICL signal in the main experiments must go *beyond* these capabilities.

| Benchmark | What it tests | Expected |
|-----------|--------------|----------|
| **Format compliance** | Can the model output a valid label (A/B) given the format? | ~100% by k=1 |
| **Retrieval (copying)** | Can it copy the label for a word it's already seen in-context? | High at k=1 |
| **Majority label** | Does it output the most frequent label in the demos? | High at low k |
| **Recency copying** | Does it output the same label as the most recent demo? | Moderate-high |

In [ ]:
BENCH_K_VALUES = [1, 2, 4, 8, 16, 32]
BENCH_N_TRIALS = 50
VALID_BINARY_LABELS = {"A", "B"}


def run_benchmarks(model, tokenizer, device, n_trials=BENCH_N_TRIALS):
    """Run all confound capability benchmarks.

    Returns a dict: benchmark_name -> {k -> list of result dicts}.
    Each result has 'accuracy' (benchmark-specific) and 'generated'.
    """
    rng = np.random.RandomState(789)
    results = defaultdict(lambda: defaultdict(list))

    for k in tqdm(BENCH_K_VALUES, desc="Benchmark k"):
        for trial in tqdm(range(n_trials), desc=f"k={k}", leave=False):

            # --- 1. FORMAT COMPLIANCE ---
            # Random A/B demos with random numbers. Score = output is a valid label.
            nums = rng.choice(1000, size=k + 1, replace=False).tolist()
            labels = [rng.choice(["A", "B"]) for _ in range(k)]
            demos = [(str(n), l) for n, l in zip(nums[:k], labels)]
            test_num = str(nums[k])
            prompt = format_demos(demos, test_num)
            res = evaluate_prompt(model, tokenizer, prompt, " A", device)
            # Check if generated text is ANY valid label (not necessarily correct)
            is_valid = 1.0 if res["generated"].strip() in VALID_BINARY_LABELS else 0.0
            results["format_compliance"][k].append({
                "accuracy": is_valid, "generated": res["generated"]
            })

            # --- 2. RETRIEVAL (COPYING) ---
            # Test word appears in the demos. Model just needs to copy its label.
            nums = rng.choice(1000, size=k + 1, replace=False).tolist()
            threshold = rng.randint(200, 801)
            demos = [(str(n), "A" if n < threshold else "B") for n in nums[:k]]
            # Pick a random demo to repeat as the test query
            repeat_idx = rng.randint(0, k)
            test_num, correct_label = demos[repeat_idx]
            prompt = format_demos(demos, test_num)
            res = evaluate_prompt(model, tokenizer, prompt, f" {correct_label}", device)
            results["retrieval"][k].append({
                "accuracy": res["accuracy"], "generated": res["generated"]
            })

            # --- 3. MAJORITY LABEL ---
            # ALL demos have the same label. Model should output that label.
            majority_label = rng.choice(["A", "B"])
            nums = rng.choice(1000, size=k + 1, replace=False).tolist()
            demos = [(str(n), majority_label) for n in nums[:k]]
            test_num = str(nums[k])
            prompt = format_demos(demos, test_num)
            res = evaluate_prompt(model, tokenizer, prompt, f" {majority_label}", device)
            results["majority_label"][k].append({
                "accuracy": res["accuracy"], "generated": res["generated"]
            })

            # --- 4. RECENCY COPYING ---
            # Score = whether model outputs the same label as the LAST demo.
            nums = rng.choice(1000, size=k + 1, replace=False).tolist()
            labels = [rng.choice(["A", "B"]) for _ in range(k)]
            demos = [(str(n), l) for n, l in zip(nums[:k], labels)]
            last_label = labels[-1]
            test_num = str(nums[k])
            prompt = format_demos(demos, test_num)
            res = evaluate_prompt(model, tokenizer, prompt, f" {last_label}", device)
            results["recency"][k].append({
                "accuracy": res["accuracy"], "generated": res["generated"]
            })

    return dict(results)


benchmark_results = run_benchmarks(model, tokenizer, device)

In [ ]:
# Plot benchmark results
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

bench_colors = {
    "format_compliance": "#2196f3",
    "retrieval": "#4caf50",
    "majority_label": "#ff9800",
    "recency": "#9c27b0",
}
bench_markers = {
    "format_compliance": "o",
    "retrieval": "s",
    "majority_label": "^",
    "recency": "D",
}
bench_labels = {
    "format_compliance": "Format compliance (output is valid A/B)",
    "retrieval": "Retrieval (copy label for repeated input)",
    "majority_label": "Majority label (all demos same label)",
    "recency": "Recency (match last demo's label)",
}

for bench_name, k_results in benchmark_results.items():
    ks = sorted(k_results.keys())
    means = [np.mean([r["accuracy"] for r in k_results[k]]) for k in ks]
    sems = [np.std([r["accuracy"] for r in k_results[k]]) / np.sqrt(len(k_results[k])) for k in ks]

    ax.errorbar(
        ks, means, yerr=sems,
        label=bench_labels.get(bench_name, bench_name),
        color=bench_colors.get(bench_name, "gray"),
        marker=bench_markers.get(bench_name, "o"),
        capsize=3, linewidth=2,
    )

ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="chance (50%)")
ax.set_xlabel("Number of demonstrations (k)")
ax.set_ylabel("Accuracy")
ax.set_title("Confound Capability Benchmarks — What the model can already do")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary
print("\nBenchmark summary (accuracy at each k):")
for bench_name in ["format_compliance", "retrieval", "majority_label", "recency"]:
    k_results = benchmark_results[bench_name]
    ks = sorted(k_results.keys())
    vals = [f"k={k}: {np.mean([r['accuracy'] for r in k_results[k]]):.0%}" for k in ks]
    print(f"  {bench_name:20s} — {', '.join(vals)}")

In [ ]:
def build_conditions(task_gen, k):
    """Build all 5 experimental conditions for a given k.

    Works with any task generator that has sample_task() and sample_distractor_task().

    Returns a list of (condition_name, prompt, target) tuples.
    Each call samples a fresh task instance so conditions share the same
    underlying mapping and test query.
    """
    if k == 0:
        _, tests = task_gen.sample_task(n_demos=0, n_test=1)
        test_word, test_label = tests[0]
        prompt = format_demos([], test_word)
        return [("standard", prompt, f" {test_label}")]

    demos, tests = task_gen.sample_task(n_demos=k, n_test=1)
    test_word, test_label = tests[0]
    target = f" {test_label}"

    # Collect the label vocabulary from this task instance
    all_labels = list(set(label for _, label in demos))

    conditions = []

    # 1. Standard: correct demos in original order
    prompt = format_demos(demos, test_word)
    conditions.append(("standard", prompt, target))

    # 2. Irrelevant demos: demos from a completely different mapping/rule
    irrel_demos = task_gen.sample_distractor_task(k)
    prompt = format_demos(irrel_demos, test_word)
    conditions.append(("irrelevant", prompt, target))

    # 3. Shuffled labels: same inputs, but labels randomly reassigned
    shuffled_labels = [all_labels[task_gen.rng.randint(0, len(all_labels))] for _ in demos]
    shuffled_demos = [(w, l) for (w, _), l in zip(demos, shuffled_labels)]
    prompt = format_demos(shuffled_demos, test_word)
    conditions.append(("shuffled_labels", prompt, target))

    # 4. Reversed order: correct demos, reversed
    reversed_demos = list(reversed(demos))
    prompt = format_demos(reversed_demos, test_word)
    conditions.append(("reversed", prompt, target))

    # 5. Recency conflict: first half correct, second half from wrong rule
    half = k // 2
    if half > 0:
        wrong_demos = task_gen.sample_distractor_task(k - half)
        conflict_demos = demos[:half] + wrong_demos
        prompt = format_demos(conflict_demos, test_word)
        conditions.append(("recency_conflict", prompt, target))

    return conditions


def run_experiment(model, tokenizer, device, task_gen, k_values, n_trials=50):
    """Run the full experiment across all k values and conditions."""
    results = defaultdict(lambda: defaultdict(list))

    for k in tqdm(k_values, desc="k values"):
        for trial in tqdm(range(n_trials), desc=f"k={k} trials", leave=False):
            conditions = build_conditions(task_gen, k)

            for cond_name, prompt, target in conditions:
                result = evaluate_prompt(model, tokenizer, prompt, target, device)
                result["k"] = k
                result["trial"] = trial
                results[cond_name][k].append(result)

    return dict(results)

In [ ]:
# 5. Results

In [ ]:
def plot_results(results, metric="loss", title_suffix="", chance_level=None):
    """Plot a metric across conditions and k values."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    colors = {
        "standard": "#2196f3",
        "irrelevant": "#f44336",
        "shuffled_labels": "#ff9800",
        "reversed": "#9c27b0",
        "recency_conflict": "#4caf50",
    }
    markers = {
        "standard": "o",
        "irrelevant": "x",
        "shuffled_labels": "s",
        "reversed": "^",
        "recency_conflict": "D",
    }

    for cond_name, k_results in results.items():
        ks = sorted(k_results.keys())
        means = [np.mean([r[metric] for r in k_results[k]]) for k in ks]
        sems = [np.std([r[metric] for r in k_results[k]]) / np.sqrt(len(k_results[k])) for k in ks]

        ax.errorbar(
            ks, means, yerr=sems,
            label=cond_name,
            color=colors.get(cond_name, "gray"),
            marker=markers.get(cond_name, "o"),
            capsize=3,
            linewidth=2,
        )

    ax.set_xlabel("Number of demonstrations (k)")
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"{metric.capitalize()} vs k {title_suffix}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    if metric == "accuracy" and chance_level is not None:
        ax.axhline(y=chance_level, color="gray", linestyle="--", alpha=0.5,
                    label=f"chance ({chance_level:.0%})")
        ax.set_ylim(-0.05, 1.05)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Full run: Task 1 (Symbol Mapping, 10-label)
# Capped at k=32 because word pool has only 50 words

task_gen_full = SymbolMappingTask(seed=123)
results_t1 = run_experiment(
    model, tokenizer, device, task_gen_full,
    k_values=[0, 1, 2, 4, 8, 16, 32],
    n_trials=50,
)
plot_results(results_t1, metric="accuracy", title_suffix="— Symbol Mapping (10-label)", chance_level=0.1)
plot_results(results_t1, metric="loss", title_suffix="— Symbol Mapping (10-label)")

## Task 2: Binary Threshold Classification

**Setup**: Numbers 0-99 are classified as "A" or "B" based on a random threshold t.
If x < t → "A", else → "B". The threshold is randomly chosen per task instance
from [20, 80] to ensure both classes are well-represented.

**Why this is better for detecting ICL**:
- **Binary** → chance is 50%, so any improvement is easy to detect
- **Learnable rule** → the model can infer the threshold from examples (not pure memorization)
- **Unlimited inputs** → supports large k (up to 256)
- **Novel each time** → different threshold per task, can't be memorized from pretraining

In [ ]:
# Full run: Task 2 (Binary Threshold Classification)
# Extended k range — binary task supports large k

task_gen_binary = BinaryThresholdTask(seed=456)
results_t2 = run_experiment(
    model, tokenizer, device, task_gen_binary,
    k_values=K_VALUES,  # [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]
    n_trials=50,
)
plot_results(results_t2, metric="accuracy", title_suffix="— Binary Threshold", chance_level=0.5)
plot_results(results_t2, metric="loss", title_suffix="— Binary Threshold")

## Task 3: Difficulty Sweep — Multi-Threshold Classification

We use `MultiThresholdTask` to sweep across **number of latent classes** N = {2, 3, 4, 6, 8, 12, 16}.
All tasks use the same structure: partition numbers 0-999 into N intervals with random boundaries.
The model must learn N-1 thresholds from demonstrations.

**Key question**: How does ICL performance degrade with task difficulty (number of classes)?

For this sweep we only run the **standard** condition (correct demos, no confound controls)
to keep runtime manageable. We plot accuracy vs k for each N, normalized against chance (1/N).

In [ ]:
# Difficulty sweep: standard condition only, varying n_classes
N_CLASSES_SWEEP = [2, 3, 4, 6, 8, 12, 16]
SWEEP_K_VALUES = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]
SWEEP_N_TRIALS = 50

difficulty_results = {}  # n_classes -> {k -> list of result dicts}

for n_classes in tqdm(N_CLASSES_SWEEP, desc="n_classes sweep"):
    task_gen = MultiThresholdTask(n_classes=n_classes, seed=100 + n_classes)
    k_results = defaultdict(list)

    for k in tqdm(SWEEP_K_VALUES, desc=f"N={n_classes}", leave=False):
        for trial in range(SWEEP_N_TRIALS):
            demos, tests = task_gen.sample_task(n_demos=k, n_test=1)
            test_word, test_label = tests[0]
            prompt = format_demos(demos, test_word)
            target = f" {test_label}"
            result = evaluate_prompt(model, tokenizer, prompt, target, device)
            result["k"] = k
            k_results[k].append(result)

    difficulty_results[n_classes] = dict(k_results)

print("Difficulty sweep complete.")

In [ ]:
# Plot 1: Raw accuracy vs k for each n_classes
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

cmap = plt.cm.viridis
colors_sweep = {n: cmap(i / (len(N_CLASSES_SWEEP) - 1)) for i, n in enumerate(N_CLASSES_SWEEP)}

for n_classes in N_CLASSES_SWEEP:
    k_results = difficulty_results[n_classes]
    ks = sorted(k_results.keys())
    means = [np.mean([r["accuracy"] for r in k_results[k]]) for k in ks]
    sems = [np.std([r["accuracy"] for r in k_results[k]]) / np.sqrt(len(k_results[k])) for k in ks]
    chance = 1.0 / n_classes

    ax.errorbar(ks, means, yerr=sems, label=f"N={n_classes} (chance={chance:.0%})",
                color=colors_sweep[n_classes], marker="o", capsize=2, linewidth=2)
    ax.axhline(y=chance, color=colors_sweep[n_classes], linestyle=":", alpha=0.3)

ax.set_xlabel("Number of demonstrations (k)")
ax.set_ylabel("Accuracy")
ax.set_title("ICL Difficulty Sweep — Accuracy vs k by Number of Classes")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9, loc="center left", bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 2: Accuracy normalized by chance (how many times better than random)
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for n_classes in N_CLASSES_SWEEP:
    k_results = difficulty_results[n_classes]
    ks = sorted(k_results.keys())
    chance = 1.0 / n_classes
    means = [np.mean([r["accuracy"] for r in k_results[k]]) / chance for k in ks]

    ax.plot(ks, means, label=f"N={n_classes}",
            color=colors_sweep[n_classes], marker="o", linewidth=2)

ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="chance (1x)")
ax.set_xlabel("Number of demonstrations (k)")
ax.set_ylabel("Accuracy / Chance")
ax.set_title("ICL Difficulty Sweep — Performance Relative to Chance")
ax.legend(fontsize=9, loc="center left", bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
print("\nMax accuracy reached (at k=256):")
for n_classes in N_CLASSES_SWEEP:
    k_results = difficulty_results[n_classes]
    max_k = max(k_results.keys())
    acc = np.mean([r["accuracy"] for r in k_results[max_k]])
    chance = 1.0 / n_classes
    print(f"  N={n_classes:2d}: acc={acc:.1%}, chance={chance:.1%}, ratio={acc/chance:.1f}x")

## Model Size Comparison

We repeat the difficulty sweep for **Pythia-160M** (smaller) and **Pythia-1.4B** (larger)
to test whether ICL capability scales with model size.

Key prediction: larger models should
1. Reach higher accuracy at the same k
2. Handle more classes before performance degrades
3. Need fewer demonstrations to saturate

We load models one at a time to avoid OOM, run the same sweep, then compare.

In [ ]:
def run_difficulty_sweep(model, tokenizer, device, n_classes_list, k_values, n_trials=50):
    """Run the difficulty sweep for a single model. Returns {n_classes -> {k -> [results]}}."""
    all_results = {}
    for n_classes in tqdm(n_classes_list, desc="n_classes"):
        task_gen = MultiThresholdTask(n_classes=n_classes, seed=100 + n_classes)
        k_results = defaultdict(list)
        for k in tqdm(k_values, desc=f"N={n_classes}", leave=False):
            for trial in range(n_trials):
                demos, tests = task_gen.sample_task(n_demos=k, n_test=1)
                test_word, test_label = tests[0]
                prompt = format_demos(demos, test_word)
                target = f" {test_label}"
                result = evaluate_prompt(model, tokenizer, prompt, target, device)
                result["k"] = k
                k_results[k].append(result)
        all_results[n_classes] = dict(k_results)
    return all_results


def load_model(model_name, device):
    """Load a model and tokenizer, return (model, tokenizer, param_count_str)."""
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)
    mdl.eval()
    n_params = sum(p.numel() for p in mdl.parameters()) / 1e6
    print(f"Loaded {model_name} ({n_params:.0f}M params)")
    return mdl, tok, f"{n_params:.0f}M"


# Store results from the 410M run we already have
scaling_results = {"410M": difficulty_results}

# --- Pythia-160M ---
print("=" * 50)
print("Running Pythia-160M...")
del model  # free memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

model_sm, tok_sm, label_sm = load_model("EleutherAI/pythia-160m", device)
scaling_results[label_sm] = run_difficulty_sweep(
    model_sm, tok_sm, device,
    n_classes_list=N_CLASSES_SWEEP,
    k_values=SWEEP_K_VALUES,
    n_trials=SWEEP_N_TRIALS,
)
del model_sm
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Pythia-1.4B ---
print("=" * 50)
print("Running Pythia-1.4B...")

model_lg, tok_lg, label_lg = load_model("EleutherAI/pythia-1.4b", device)
scaling_results[label_lg] = run_difficulty_sweep(
    model_lg, tok_lg, device,
    n_classes_list=N_CLASSES_SWEEP,
    k_values=SWEEP_K_VALUES,
    n_trials=SWEEP_N_TRIALS,
)
del model_lg
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Reload 410M for any further experiments
model, tokenizer, _ = load_model("EleutherAI/pythia-410m", device)

print("\nModel scaling sweep complete.")

In [ ]:
# Plot 1: Accuracy at k=256 vs n_classes, for each model size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_colors = {"160M": "#f44336", "410M": "#ff9800", "1393M": "#4caf50", "1.4B": "#4caf50"}
model_markers = {"160M": "s", "410M": "o", "1393M": "^", "1.4B": "^"}
# Normalize label names (the param count might vary slightly)
model_labels_sorted = sorted(scaling_results.keys(),
                              key=lambda x: float(x.replace("M", "").replace("B", "000")))

# --- Panel 1: Raw accuracy at k=256 ---
ax = axes[0]
for model_label in model_labels_sorted:
    diff_results = scaling_results[model_label]
    n_classes_list = sorted(diff_results.keys())
    accs = []
    for n in n_classes_list:
        max_k = max(diff_results[n].keys())
        acc = np.mean([r["accuracy"] for r in diff_results[n][max_k]])
        accs.append(acc)

    color = model_colors.get(model_label, "gray")
    marker = model_markers.get(model_label, "o")
    ax.plot(n_classes_list, accs, label=f"Pythia-{model_label}",
            color=color, marker=marker, linewidth=2, markersize=8)

# Plot chance level
n_range = sorted(list(scaling_results.values())[0].keys())
ax.plot(n_range, [1.0/n for n in n_range], label="chance (1/N)",
        color="gray", linestyle="--", alpha=0.5)

ax.set_xlabel("Number of classes (N)")
ax.set_ylabel("Accuracy at k=256")
ax.set_title("ICL Performance vs Task Difficulty by Model Size")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# --- Panel 2: Chance-normalized accuracy at k=256 ---
ax = axes[1]
for model_label in model_labels_sorted:
    diff_results = scaling_results[model_label]
    n_classes_list = sorted(diff_results.keys())
    ratios = []
    for n in n_classes_list:
        max_k = max(diff_results[n].keys())
        acc = np.mean([r["accuracy"] for r in diff_results[n][max_k]])
        ratios.append(acc / (1.0 / n))

    color = model_colors.get(model_label, "gray")
    marker = model_markers.get(model_label, "o")
    ax.plot(n_classes_list, ratios, label=f"Pythia-{model_label}",
            color=color, marker=marker, linewidth=2, markersize=8)

ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="chance (1x)")
ax.set_xlabel("Number of classes (N)")
ax.set_ylabel("Accuracy / Chance")
ax.set_title("Relative ICL Performance vs Difficulty by Model Size")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Plot 3: Learning curves for selected difficulties, all models ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
selected_n = [2, 8, 16]

for idx, n_classes in enumerate(selected_n):
    ax = axes[idx]
    chance = 1.0 / n_classes

    for model_label in model_labels_sorted:
        diff_results = scaling_results[model_label]
        if n_classes not in diff_results:
            continue
        k_results = diff_results[n_classes]
        ks = sorted(k_results.keys())
        means = [np.mean([r["accuracy"] for r in k_results[k]]) for k in ks]
        sems = [np.std([r["accuracy"] for r in k_results[k]]) / np.sqrt(len(k_results[k])) for k in ks]

        color = model_colors.get(model_label, "gray")
        marker = model_markers.get(model_label, "o")
        ax.errorbar(ks, means, yerr=sems, label=f"Pythia-{model_label}",
                    color=color, marker=marker, capsize=2, linewidth=2)

    ax.axhline(y=chance, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Number of demonstrations (k)")
    ax.set_title(f"N={n_classes} classes (chance={chance:.0%})")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.set_ylabel("Accuracy")
    ax.legend(fontsize=9)

plt.suptitle("ICL Learning Curves by Model Size", fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
print("\nAccuracy at k=256 by model size and n_classes:")
header = f"{'N':>4s}" + "".join(f"  {ml:>8s}" for ml in model_labels_sorted) + "   chance"
print(header)
for n in sorted(list(scaling_results.values())[0].keys()):
    row = f"{n:4d}"
    for ml in model_labels_sorted:
        if n in scaling_results[ml]:
            max_k = max(scaling_results[ml][n].keys())
            acc = np.mean([r["accuracy"] for r in scaling_results[ml][n][max_k]])
            row += f"  {acc:>7.1%}"
        else:
            row += f"  {'N/A':>7s}"
    row += f"  {1.0/n:>6.1%}"
    print(row)

In [ ]:
# 6. Interpretation

# Placeholder — fill in after seeing results.
# Key questions to address:
#
# 1. Does accuracy increase / loss decrease with k for the STANDARD condition?
#    -> If yes: evidence of ICL. If no: model may be too small.
#
# 2. Does the IRRELEVANT condition also improve with k?
#    -> If yes: "more context helps" confound, not true ICL.
#    -> If no: improvement is specific to relevant demos.
#
# 3. Does SHUFFLED LABELS perform like standard or like irrelevant?
#    -> Like standard: model is imitating format, not learning the mapping.
#    -> Like irrelevant: model needs correct labels to improve.
#
# 4. Does REVERSED perform similarly to standard?
#    -> If yes: order doesn't matter much (Bayesian-like).
#    -> If worse: recency/position effects.
#
# 5. RECENCY CONFLICT: does the model follow the correct (early) or wrong (recent) demos?
#    -> Following recent: recency bias dominates.
#    -> Following early: robust evidence integration.